# DIC予測 v2：全結合NNだけで組み直し、入力列を見直す

2周目の版。方針と実験の詳細は戦略メモの「2週目」タブに書いた。このノートでは次の順に進める。

1. データ確認：列名をそろえ（`TA1.x`→`TA1`）、欠損・重複列・id順を確認する
2. 前処理とモデルの定義：前処理はnumpyの計算だけで書く。モデルは `Linear→活性化→Linear→活性化→Linear` の全結合NNだけ
3. 交差検証：1周目と同じ手順で、基準Pと採用案（S・G・S+G）を比べる
4. 全train（1,454行）で学習し直して `submission.csv` を作る

1周目（v1）から変えた点
- 線形回帰（`np.linalg.lstsq`）による学習側の外れ値除外をやめた。全結合NN以外のモデルを使わないため
- 入力→出力の線形スキップをやめた。モデルを直列の全結合層だけにするため
- 入力から `NH3uM` を外した（14列→13列）。外した方が交差検証の誤差が小さかった
- 活性化関数だけが違う2種類（SiLU・GELU）の全結合NNの予測を平均する

守ること
- モデルは全結合NNだけ。予測の平均も、全結合NNどうしの単純平均だけ
- 前処理は列の選択、平均・標準偏差による標準化、`log1p`、`clip` だけ。scikit-learnなどの前処理ライブラリは使わない。ドメイン特徴・交差特徴も使わない
- 平均・標準偏差・クリップ範囲は、すべて学習部分だけから求める
- 評価は元のDIC単位のRMSE。各行を学習に使わなかったモデルの予測（OOF）を全1,454行分集めて計算する
- 評価用のfoldは学習にも学習回数の決定にも使わない（学習回数は固定）

実行時間の目安：CPU 2コアで交差検証が約25分、補助評価が約1分、全件学習が約40秒。`QUICK = True` にすると分割1通り・seed1つで試せる。`EVAL_FRESH = False` にすると確認用の分割を省いて約半分になる。

保存されている出力は、Claudeの作業環境（Linux、CPU 2コア）でこのノートのコードをそのまま実行したもの（2026年11月20日）。Colabで再実行すると、ライブラリの版によって小数点以下が少し変わることがある。

## 引用元・参考資料

このノートのコード・設定が何に基づくかをまとめる。本文中でも、使った箇所のコメントに番号で示す。

**もとにしたノート・記録**
- [A] 配布ノートブック（ReLU 128→64、DICを生値で学習）：出発点にしたコード。<https://colab.research.google.com/drive/10tBmg7J6_fDt0KLdTTG2oYDSPAKHf2y9>
- [B] 戦略メモ「機械学習基礎 コンペ戦略メモ」：1周目タブ・戦略1（ChatGPT）タブ・戦略2（Claude）タブ・2週目タブ。<https://docs.google.com/document/d/1U_WHC9poxdGDOFZp6pdwKSpunfI8yoDc2RCIthOKS2M/edit>
  - 1周目から引き継いだもの：`TA1.x`→`TA1` の改名、`R_Nuts`・`R_TEMP` の除外、5列の `log1p`、入力とDICの標準化、SiLU 128→64、Huber損失（δ=0.05）、Adam 3e-3からのcosine学習率・300 epoch固定、予測時の入力クリップ、OOFによる評価手順
  - このノートの前の版（v1）は、戦略1（ChatGPTとの検討）と戦略2（Claudeとの検討）を同じ条件で比べたもの。v2のコードと実験はClaude（Anthropic）と作成した
- [C] 参考にしたKaggleノートブック：T.Msk（taishinmasaki）「notebook92228211c8」Version 1、Apache 2.0ライセンス、Private Score 3.80892。<https://www.kaggle.com/code/taishinmasaki/notebook92228211c8/notebook?scriptVersionId=370684486>
  - 参考にした点：256×4→128の深い全結合NN、AdamW（重み減衰0.1）、OneCycleLR、500 epoch、第1層だけ重み減衰を強める案を、こちらの入力・評価手順で試した。分割1通りで4.995（同じ分割の基準は5.015）と差が小さく、学習時間が10倍以上になるため採用していない
  - コードはコピーしていない。SiLU・Huber損失（δ=0.05）・DICの標準化・seed平均は[C]にも含まれるが、このノートでは1周目[B]から引き継いだ

**データ**
- [D] Kaggle「EDS 232 Ocean chemistry prediction for CalCOFI」のtrain.csv・test.csv・sample_submission.csv。<https://www.kaggle.com/competitions/eds-232-ocean-chemistry-prediction-for-calcofi-2/data>
  - このノートは同じ3ファイルをGitHubのコピーから読む：<https://github.com/Toyoda05/Kaggle-eds232>
- [E] 列の意味・単位：CalCOFI Bottle Database。<https://calcofi.org/data/oceanographic-data/bottle-database/>

**手法**
- [F] Adam：D. P. Kingma, J. Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015（arXiv:1412.6980）。実装は `torch.optim.Adam`
- [G] SiLU：S. Elfwing, E. Uchibe, K. Doya, "Sigmoid-Weighted Linear Units for Neural Network Function Approximation in Reinforcement Learning", Neural Networks, 2018（arXiv:1702.03118）。実装は `torch.nn.SiLU` <https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html>
- [H] GELU：D. Hendrycks, K. Gimpel, "Gaussian Error Linear Units (GELUs)", 2016（arXiv:1606.08415）。実装は `torch.nn.GELU`
- [I] Huber損失：P. J. Huber, "Robust Estimation of a Location Parameter", Annals of Mathematical Statistics, 35(1), 1964。実装は `torch.nn.HuberLoss` <https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.loss.HuberLoss.html>
- [J] cosine学習率：I. Loshchilov, F. Hutter, "SGDR: Stochastic Gradient Descent with Warm Restarts", ICLR 2017（arXiv:1608.03983）。実装は `torch.optim.lr_scheduler.CosineAnnealingLR`（再始動なしで使う）
- [K] ライブラリ：PyTorch（モデル・学習）、NumPy・pandas（読み込みと数値計算）。前処理用のライブラリは使っていない

In [12]:
# 0. 設定・データ読み込み（GitHubのRawデータから読み込みます）［データの出どころは引用元D］
import os, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

QUICK = False          # True: 分割seed1通り×学習seed1つで動作確認だけ行う
EVAL_FRESH = True      # 探索に使っていない分割（確認用）でも評価するか
RUN_GROUP_CV = True    # 補助評価（座標ごとの分割）を実行するか

OUT_DIR = '/content/dic_v2'
os.makedirs(OUT_DIR, exist_ok=True)

# GitHubのRaw URL
BASE_URL = 'https://raw.githubusercontent.com/Toyoda05/Kaggle-eds232/main/'

train = pd.read_csv(BASE_URL + 'train.csv')
test = pd.read_csv(BASE_URL + 'test.csv')
sub = pd.read_csv(BASE_URL + 'sample_submission.csv')

torch.set_num_threads(1)   # 小さいNNを1個ずつ学習するので、1スレッドの方が速い
print('torch', torch.__version__, '| CPU', os.cpu_count())
print('train', train.shape, 'test', test.shape, 'sample', sub.shape)

torch 2.14.1+cu130 | CPU 2
train (1454, 19) test (485, 17) sample (485, 2)


In [13]:
# 1. データ確認：列をそろえ、前提が崩れていないことを assert で確かめる（1周目と同じ。引用元B）
tr = train.drop(columns=['Unnamed: 12']).rename(columns={'TA1.x': 'TA1'})
te = test.copy()

INPUT_ALL = [c for c in tr.columns if c not in ('id', 'DIC')]
assert INPUT_ALL == [c for c in te.columns if c != 'id'], 'trainとtestの入力列（名前・順序）が一致しません'
assert len(INPUT_ALL) == 16
assert tr[INPUT_ALL + ['DIC']].notna().all().all() and te[INPUT_ALL].notna().all().all(), '欠損があります'
assert (tr['NH3uM'] == tr['R_Nuts']).all() and (te['NH3uM'] == te['R_Nuts']).all()
assert np.array_equal(te['id'].values, sub['id'].values), 'testとsample_submissionのid順が違います'

y_all = tr['DIC'].to_numpy(dtype=np.float64)
N = len(tr)
# NO2uMが極端に大きい2行（6.19・8.19）。除外はせず、参考の集計だけに使う
EXTREME_IDS = [702, 854]
is_extreme = tr['id'].isin(EXTREME_IDS).to_numpy()
# 補助評価用：緯度・経度が完全に一致する行を同じグループにする
coord_group = tr.groupby(['Lat_Dec', 'Lon_Dec'], sort=False).ngroup().to_numpy()

print('入力候補16列:', INPUT_ALL)
print('欠損なし／NH3uM=R_Nuts（全行）／id順一致 を確認')
print('DIC 平均 %.2f, 標準偏差 %.2f' % (y_all.mean(), y_all.std()))
print('NH3uMが0の行: train %d / %d, test %d / %d' % ((tr.NH3uM == 0).sum(), N, (te.NH3uM == 0).sum(), len(te)))
print('座標の数（train）:', coord_group.max() + 1)

入力候補16列: ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_TEMP', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Nuts', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']
欠損なし／NH3uM=R_Nuts（全行）／id順一致 を確認
DIC 平均 2150.47, 標準偏差 113.12
NH3uMが0の行: train 636 / 1454, test 231 / 485
座標の数（train）: 259


## 2. 前処理と候補

前処理は次の4つだけで、すべてnumpyの計算で書く。
1. 列の選択：`R_Nuts`（`NH3uM` と全行同じ）と `R_TEMP`（`Temperature_degC` との相関0.99998）を外す。採用案ではさらに `NH3uM` を外す
2. `log1p`：値が右に偏った列（`NO2uM`、`R_Depth`、`R_DYNHT`、`SiO3uM`、残す場合は `NH3uM`）
3. 標準化：入力とDICを、学習部分の平均・標準偏差で `(値−平均)÷標準偏差` にする。予測は元のDIC単位に戻す
4. クリップ：予測時に、標準化後の各入力を学習部分の最小〜最大に収める（学習範囲の外へ外挿しない）

| 名前 | 入力 | 活性化関数 | 位置づけ |
| --- | --- | --- | --- |
| P | 14列 | SiLU | 基準。1周目の採用案CLから、線形回帰による外れ値除外と線形スキップを外したもの |
| S | 13列（`NH3uM` なし） | SiLU | 採用案の1つ目 |
| G | 13列（`NH3uM` なし） | GELU | 採用案の2つ目 |

共通：全結合NN 128→64、Huber損失（δ=0.05）、Adam（学習率3e-3から0までcosineで下げる）、300 epoch、batch 64、重み減衰・Dropoutなし。

採用案は `S+G`（SとGの予測を1:1で平均。各3seedで計6モデル）。重みは固定で、探索していない。
`NH3uM` を外す理由：trainの44%（636行）が0で、値のある行も0〜2.75と小さい。外した方が、探索用・確認用どちらの分割でも誤差が小さかった（詳細は戦略メモ「2週目」タブ）。

In [14]:
# 2. 候補の定義と前処理（numpyの計算だけで書く）
COMMON = dict(hidden=(128, 64), batch=64, lr=3e-3, epochs=300, delta=0.05)
LOG1P_COLS = ['NO2uM', 'NH3uM', 'R_Depth', 'R_DYNHT', 'SiO3uM']   # 1周目と同じ5列（引用元B）。入力に残っている列だけ変換する
CONFIGS = {
    'P': dict(drop=['R_Nuts', 'R_TEMP'], act='silu'),
    'S': dict(drop=['R_Nuts', 'R_TEMP', 'NH3uM'], act='silu'),
    'G': dict(drop=['R_Nuts', 'R_TEMP', 'NH3uM'], act='gelu'),
}
BLENDS = {
    'P':   {'P': 1.0},
    'S':   {'S': 1.0},
    'G':   {'G': 1.0},
    'S+G': {'S': 0.5, 'G': 0.5},
}
ADOPTED = 'S+G'
SPLIT_SEEDS = [2026, 2027, 100]   # 1周目と同じ分割seed（探索にも使った）
FRESH_SEEDS = [11, 22, 33]        # 確認用：構成を決めるまで使わなかった分割seed
TRAIN_SEEDS = [17, 42, 2026]      # 学習seed（全候補で同じ3つ）
if QUICK:
    SPLIT_SEEDS, FRESH_SEEDS, TRAIN_SEEDS = [2026], [11], [42]
ALL_SPLITS = SPLIT_SEEDS + (FRESH_SEEDS if EVAL_FRESH else [])


def feature_cols(cfg):
    """列の選択"""
    return [c for c in INPUT_ALL if c not in cfg['drop']]


def make_X(df, cfg):
    """列の選択と log1p。標準化とクリップは学習部分の値が必要なので fit_predict の中で行う"""
    X = df[feature_cols(cfg)].astype(np.float64).copy()
    for c in LOG1P_COLS:
        if c in X.columns:
            X[c] = np.log1p(X[c])
    return X.to_numpy()


def fit_scaler(X):
    """標準化に使う平均・標準偏差（学習部分だけから求める）"""
    mean, std = X.mean(axis=0), X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return mean, std


def standardize(X, mean, std):
    return (X - mean) / std


for name, cfg in CONFIGS.items():
    print(f'{name}: {len(feature_cols(cfg))}列', feature_cols(cfg))

P: 14列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']
S: 13列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']
G: 13列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']


In [ ]:
# 3. モデル・学習関数
ACTIVATIONS = {'silu': nn.SiLU, 'gelu': nn.GELU}   # 引用元G・H


def make_mlp(n_in, hidden, act):
    """全結合NN：Linear→活性化→Linear→活性化→Linear（出力は1ノード・活性化なし）。全結合層を直列に並べるだけ"""
    layers, d = [], n_in
    for h in hidden:
        layers += [nn.Linear(d, h), ACTIVATIONS[act]()]
        d = h
    layers.append(nn.Linear(d, 1))
    return nn.Sequential(*layers)


def fit_predict(cfg, Xtr, ytr, X_list, seed):
    """学習部分(Xtr, ytr)だけで前処理値を求めて学習し、X_listの各行列を元のDIC単位で予測する"""
    mean, std = fit_scaler(Xtr)
    y_mean, y_std = ytr.mean(), ytr.std()
    Xt = torch.tensor(standardize(Xtr, mean, std), dtype=torch.float32)
    yt = torch.tensor((ytr - y_mean) / y_std, dtype=torch.float32)

    torch.manual_seed(seed)
    model = make_mlp(Xt.shape[1], COMMON['hidden'], cfg['act'])
    opt = torch.optim.Adam(model.parameters(), lr=COMMON['lr'])                       # 引用元F
    batch, epochs = COMMON['batch'], COMMON['epochs']
    steps = int(np.ceil(len(Xt) / batch))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)     # 引用元J
    crit = nn.HuberLoss(delta=COMMON['delta'])                                        # 引用元I
    g = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        idx = torch.randperm(len(Xt), generator=g)
        for s in range(steps):
            b = idx[s * batch:(s + 1) * batch]
            opt.zero_grad()
            loss = crit(model(Xt[b]).squeeze(-1), yt[b])
            loss.backward()
            opt.step()
            sched.step()
    model.eval()
    lo, hi = Xt.min(0).values.numpy(), Xt.max(0).values.numpy()   # クリップ範囲（学習部分の最小・最大）
    preds = []
    with torch.no_grad():
        for X in X_list:
            Z = np.clip(standardize(X, mean, std), lo, hi)        # 学習部分の範囲外へ外挿しない
            out = model(torch.tensor(Z, dtype=torch.float32)).squeeze(-1).numpy().astype(np.float64)
            preds.append(out * y_std + y_mean)                    # 元のDIC単位に戻す
    return preds


def rmse(p, y=y_all, mask=None):
    if mask is not None:
        p, y = p[mask], y[mask]
    return float(np.sqrt(np.mean((p - y) ** 2)))


print(make_mlp(13, COMMON['hidden'], 'silu'))
# 動作確認（1回だけ学習して時間を測る）
_t = time.time()
_cfg = CONFIGS['S']
_X = make_X(tr, _cfg)
_p = fit_predict(_cfg, _X[:1163], y_all[:1163], [_X[1163:]], seed=0)[0]
print('動作確認 RMSE %.3f / %.1f秒' % (rmse(_p, y_all[1163:]), time.time() - _t))

Sequential(
  (0): Linear(in_features=13, out_features=128, bias=True)
  (1): SiLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): SiLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
)


In [ ]:
# 4. 分割と交差検証（OOF予測はファイルに保存し、途中で止まっても再開できる。手順は1周目と同じ。引用元B）
def make_folds(split_seed, groups=None, k=5):
    """各行のfold番号を返す。groupsを渡すと同じグループは必ず同じfoldに入る"""
    rng = np.random.RandomState(split_seed)
    fold = np.empty(N, dtype=int)
    if groups is None:
        for f, part in enumerate(np.array_split(rng.permutation(N), k)):
            fold[part] = f
    else:
        names = np.unique(groups)
        for f, part in enumerate(np.array_split(rng.permutation(names), k)):
            fold[np.isin(groups, part)] = f
    return fold


def run_cv(name, split_seed, seeds, split_type='random'):
    """候補nameのOOF予測（学習seedごと）を返す。shape = (len(seeds), N)"""
    cfg = CONFIGS[name]
    X = make_X(tr, cfg)
    fold = make_folds(split_seed, coord_group if split_type == 'group' else None)
    out = np.zeros((len(seeds), N))
    for i, seed in enumerate(seeds):
        path = os.path.join(OUT_DIR, f'oof_{split_type}{split_seed}_{name}_seed{seed}.npy')
        if os.path.exists(path):
            out[i] = np.load(path)
            continue
        for f in range(fold.max() + 1):
            a, b = fold != f, fold == f
            out[i, b] = fit_predict(cfg, X[a], y_all[a], [X[b]], seed=seed * 100 + f)[0]
        np.save(path, out[i])
    return out


def blend(oofs, weights):
    """各候補の学習seed平均を、固定の重みで混ぜる"""
    return sum(w * oofs[n].mean(0) for n, w in weights.items())

In [ ]:
# 5. 主評価：ランダム5分割 × 学習seed3つで、P・S・GのOOFを作る
t0 = time.time()
OOF = {}
for ss in ALL_SPLITS:
    OOF[ss] = {}
    for name in CONFIGS:
        t1 = time.time()
        OOF[ss][name] = run_cv(name, ss, TRAIN_SEEDS)
        print(f'split {ss:>4} {name}: seed平均OOF RMSE {rmse(OOF[ss][name].mean(0)):.4f}'
              f'（{time.time() - t1:.0f}秒）')
print(f'合計 {time.time() - t0:.0f}秒')

In [ ]:
# 6. 集計：候補ごと・分割ごとのOOF RMSE（全行／極端な2行を除く）
rows = []
for ss in ALL_SPLITS:
    for name in CONFIGS:
        for i, seed in enumerate(TRAIN_SEEDS):
            rows.append(dict(split=ss, candidate=name, kind=f'単体 seed{seed}', rmse=rmse(OOF[ss][name][i]),
                             rmse_wo2=rmse(OOF[ss][name][i], mask=~is_extreme)))
    for bname, w in BLENDS.items():
        p = blend(OOF[ss], w)
        rows.append(dict(split=ss, candidate=bname, kind='平均（3seed）', rmse=rmse(p), rmse_wo2=rmse(p, mask=~is_extreme)))
res = pd.DataFrame(rows)


def make_table(value, kind_is_blend=True):
    d = res[(res.kind == '平均（3seed）') == kind_is_blend]
    t = d.groupby(['candidate', 'split'])[value].mean().unstack()[ALL_SPLITS]
    t['探索用の平均'] = t[SPLIT_SEEDS].mean(axis=1)
    if EVAL_FRESH:
        t['確認用の平均'] = t[FRESH_SEEDS].mean(axis=1)
    return t


table = make_table('rmse')
table_wo2 = make_table('rmse_wo2')
single = make_table('rmse', kind_is_blend=False)
print('【主指標】OOF RMSE（全1,454行、3seedの予測平均、元のDIC単位）')
display(table.loc[list(BLENDS)].round(4))
print('【参考】極端な2行（id 702・854）を除いたRMSE')
display(table_wo2.loc[list(BLENDS)].round(4))
print('【参考】単体モデル（1seed）のOOF RMSEの平均')
display(single.round(4))
print(f'採用: {ADOPTED} | 探索用の平均 {table.loc[ADOPTED, "探索用の平均"]:.4f}（基準P {table.loc["P", "探索用の平均"]:.4f}）'
      + (f' | 確認用の平均 {table.loc[ADOPTED, "確認用の平均"]:.4f}（基準P {table.loc["P", "確認用の平均"]:.4f}）' if EVAL_FRESH else ''))
res.to_csv(os.path.join(OUT_DIR, 'cv_results.csv'), index=False)

In [ ]:
# 7. 補助評価：同じ緯度・経度の行を同じfoldにまとめた5分割（分割seed 2026・学習seed 42）
#    未知地点への弱さを見るための値。主指標とは平均せず、採否にも使わない
if RUN_GROUP_CV:
    t0 = time.time()
    G = {name: run_cv(name, 2026, [42], split_type='group') for name in CONFIGS}
    R = {name: run_cv(name, 2026, [42]) for name in CONFIGS}
    rows_g = []
    for bname, w in BLENDS.items():
        pg = sum(wt * G[n][0] for n, wt in w.items())
        pr = sum(wt * R[n][0] for n, wt in w.items())
        rows_g.append(dict(candidate=bname, random_rmse=rmse(pr), group_rmse=rmse(pg),
                           group_rmse_wo2=rmse(pg, mask=~is_extreme)))
    group_table = pd.DataFrame(rows_g).set_index('candidate')
    print(f'学習seed 42のみ・分割seed 2026（{time.time() - t0:.0f}秒）')
    display(group_table.round(4))

## 3. 提出ファイルの作成

採用案 `S+G` の各候補を、全train 1,454行・学習seed 3つで学習し直す（学習回数は300 epochのまま。検証用を分けないので早期終了は使わない）。
前処理値（平均・標準偏差・クリップ範囲）も全trainから求め直す。元のDIC単位に戻した6モデルの予測を平均する。

In [ ]:
# 8. 全trainで再学習して submission.csv を作る
t0 = time.time()
test_pred = {}
for name in BLENDS[ADOPTED]:
    cfg = CONFIGS[name]
    X, Xte = make_X(tr, cfg), make_X(te, cfg)
    test_pred[name] = np.mean([fit_predict(cfg, X, y_all, [Xte], seed=seed)[0] for seed in [17, 42, 2026]], axis=0)
pred = sum(w * test_pred[n] for n, w in BLENDS[ADOPTED].items())

submission = pd.DataFrame({'id': te['id'].values, 'DIC': pred})
assert len(submission) == 485 and submission.columns.tolist() == ['id', 'DIC']
assert submission['DIC'].notna().all() and np.isfinite(submission['DIC']).all()
assert np.array_equal(submission['id'].values, sub['id'].values)
submission.to_csv('submission.csv', index=False)

record = dict(adopted=ADOPTED, weights=BLENDS[ADOPTED], common=COMMON, log1p=LOG1P_COLS,
              configs={n: CONFIGS[n] for n in BLENDS[ADOPTED]},
              split_seeds=SPLIT_SEEDS, fresh_seeds=FRESH_SEEDS if EVAL_FRESH else [], train_seeds=TRAIN_SEEDS,
              oof_rmse={str(k): round(float(v), 5) for k, v in table.loc[ADOPTED, ALL_SPLITS].items()},
              oof_rmse_mean_search=round(float(table.loc[ADOPTED, '探索用の平均']), 5),
              oof_rmse_mean_fresh=round(float(table.loc[ADOPTED, '確認用の平均']), 5) if EVAL_FRESH else None)
with open(os.path.join(OUT_DIR, 'selection.json'), 'w', encoding='utf-8') as fp:
    json.dump(record, fp, ensure_ascii=False, indent=2)
print(json.dumps(record, ensure_ascii=False, indent=1))
print(f'全件学習 {time.time() - t0:.0f}秒 | test予測 平均 {pred.mean():.2f}, 範囲 {pred.min():.2f}〜{pred.max():.2f}')
print('SとGのtest予測の差（RMS）: %.3f' % float(np.sqrt(np.mean((test_pred['S'] - test_pred['G']) ** 2))))
display(submission.head())
# ダウンロードする場合（Colab以外では何もしない）
try:
    from google.colab import files
    files.download('submission.csv')
except ImportError:
    pass

In [ ]:
# score: 3.68458
# name: 豊田大輝
# id: 62512697